# Session 11: Advanced Retrieval with LangChain

## Learning Objectives:

- Understand and implement multiple retrieval strategies for RAG
- Compare naive, BM25, multi-query, parent-document, contextual compression, ensemble, and semantic chunking approaches
- Build RAG chains over a health and wellness knowledge base using LangChain and QDrant

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

---

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

> NOTE: Create a `.env` file in this directory with `OPENAI_API_KEY` and `COHERE_API_KEY` to avoid being prompted each time.

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Health and Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, stress management, habits, and common health concerns.

### Data Preparation

We'll load the wellness guide as a single document, then split it into smaller chunks using a `RecursiveCharacterTextSplitter` for our vector store. We also keep the raw (unsplit) document for use with the Parent Document Retriever and Semantic Chunker later.

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/HealthWellnessGuide.txt")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
wellness_docs = text_splitter.split_documents(raw_docs)

Let's verify our data was loaded and split correctly!

In [4]:
print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(wellness_docs)}")
print(f"\nExample chunk:\n{wellness_docs[0]}")

Raw documents: 1
Split chunks: 45

Example chunk:
page_content='The Personal Wellness Guide
A Comprehensive Resource for Health and Well-being

PART 1: EXERCISE AND MOVEMENT

Chapter 1: Understanding Exercise Basics

Exercise is one of the most important things you can do for your health. Regular physical activity can improve your brain health, help manage weight, reduce the risk of disease, strengthen bones and muscles, and improve your ability to do everyday activities.' metadata={'source': 'data/HealthWellnessGuide.txt'}


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "wellness_guide".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    wellness_docs,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide",
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [8]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [9]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [10]:
naive_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n1. Cat-Cow Stretch: Start on your hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Perform 10-15 repetitions.\n\n2. Bird Dog: From hands and knees, extend the opposite arm and leg while keeping your core engaged. Hold each extension for about 5 seconds, then switch sides. Do 10 repetitions per side.\n\n3. Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis. Hold for 10 seconds and repeat 8-12 times.\n\nThese exercises are recommended to help alleviate lower back discomfort and may also help prevent future episodes. As always, consult with a healthcare professional before starting any new exercise routine, especially if you have existing health conditions.'

In [11]:
naive_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical repair, immune function, mental well-being, and cognitive processes. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep (7-9 hours for adults) is crucial for maintaining a healthy immune system, managing stress, and ensuring proper functioning of the brain and body. Poor sleep or sleep disorders like insomnia can negatively impact these processes, leading to health issues. Therefore, practicing good sleep hygiene and creating an optimal sleep environment are important strategies to enhance overall health.'

In [12]:
naive_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking water to stay hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of the temples and neck\n- Using essential oils like peppermint or lavender\n- Maintaining a regular sleep schedule\n- Practicing deep breathing exercises\n- Doing progressive muscle relaxation\n- Engaging in grounding techniques by naming things you see, hear, feel, smell, or taste\n- Taking short walks, preferably in nature\n- Listening to calming music\n\nThese methods can help alleviate headache symptoms and reduce stress naturally.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [13]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(wellness_docs)

We'll construct the same chain - only changing the retriever.

In [14]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [15]:
bm25_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, then alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend the opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Perform 10 repetitions per side.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds and repeat 8-12 times.\n\nThese gentle exercises can help alleviate discomfort and prevent future episodes of lower back pain.'

In [16]:
bm25_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

"Sleep has a significant impact on overall health. Adults generally need 7-9 hours of sleep per night, and proper sleep is essential for the body's physical and mental well-being. During sleep, the body goes through cycles of about 90 minutes, including REM and non-REM stages. The stages of sleep play different roles: light sleep helps you transition into deeper sleep, which is critical for physical repair and regeneration, while REM sleep is important for brain activity, memory, and learning. \n\nAdequate sleep supports the immune system, mental health, and overall function. Poor sleep or sleep disturbances like insomnia can impair these processes, leading to health issues. Creating a sleep-friendly environment—such as maintaining a comfortable temperature, ensuring darkness, and minimizing noise—can promote better sleep quality. Overall, good sleep is foundational to maintaining and improving overall health."

In [17]:
bm25_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress include practicing relaxation techniques such as meditation and deep breathing exercises, engaging in progressive muscle relaxation, and using herbal teas like chamomile or valerian root. For headaches related to stress, staying well-hydrated, managing triggers like eye strain and poor sleep, and practicing relaxation methods can help alleviate symptoms. Always consult a healthcare provider if symptoms persist or worsen.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer:


In the two examples above, we did not get better answers using BM25.

According to the theory, an example of a query where BM25 is better than embeddings is one that depends on exact keywords/phrases, rare terms, acronyms, numbers, or “exact match” signals.


Maybe it would be better to ask, for example, what are the side effects of taking paracetamol.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [18]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [19]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [20]:
contextual_compression_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on your hands and knees, then alternate between arching your back up (like a cat) and letting it sag down (like a cow). Repeat 10-15 times.\n- Bird Dog: From your hands and knees, extend one arm and the opposite leg while keeping your core engaged. Hold for about 5 seconds, then switch sides. Do 10 repetitions per side.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your lower back against the floor by tightening your abdominal muscles and tilting your pelvis slightly. Hold for 10 seconds and repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises can help alleviate lower back discomfort and prevent future episodes.'

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep is essential for overall health because it supports physical repair, mental well-being, and cognitive functions. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep, typically 7-9 hours per night for adults, also helps maintain a healthy immune system and emotional stability. Poor sleep or sleep disorders like insomnia can negatively impact health, making proper sleep habits and an optimal sleep environment important for overall wellness.'

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing deep breathing exercises, progressive muscle relaxation, and grounding techniques. Taking a short walk in nature or listening to calming music can also help relieve stress. For headaches specifically, staying hydrated by drinking water, applying cold or warm compresses, resting in a dark, quiet room, gently massaging the temples and neck, using peppermint or lavender essential oils, and maintaining a regular sleep schedule are recommended.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [23]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [24]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [25]:
multi_query_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees. Alternate between arching your back up (cat position) and letting it sag down (cow position). Do 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend your opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Complete 10 repetitions per side.\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over your chest, tighten abdominal muscles, and raise your shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- **Pelvic Tilts:** Lie on your back with knees bent. Flatten your back against the floor by tightening your abs and tilting your pelvis upward. Hold for 10 seconds, then repeat 8-12 times.\n\nThese exercises, along with gentle stretch

In [26]:
multi_query_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical repair, mental well-being, and cognitive functions. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate and quality sleep (7-9 hours per night) is essential for maintaining a healthy immune system, reducing stress, and improving mood. Poor sleep can lead to problems such as increased stress, weakened immunity, cognitive difficulties, and a higher risk of chronic conditions. Therefore, practicing good sleep hygiene and ensuring sufficient restful sleep are crucial for overall health.'

In [27]:
multi_query_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress include practicing deep breathing, progressive muscle relaxation, grounding techniques, taking short walks in nature, and listening to calming music. To help alleviate headaches naturally, you can stay hydrated by drinking water, apply cold or warm compresses to the head or neck, rest in a dark and quiet room, gently massage your temples and neck, use essential oils like peppermint or lavender, and maintain a regular sleep schedule.'

### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### Answer:

Generating multiple reformulations of a user query improves recall because a single query may not match the exact wording used in relevant documents.

When an LLM creates several paraphrased versions of the same question, each version may use different vocabulary, synonyms, or focus on different aspects of the problem.


Example from above : "Sleep significantly impacts overall health by supporting physical, mental, and cognitive functions. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep (typically 7-9 hours for adults) is crucial for maintaining immune function, reducing stress, and promoting recovery from illnesses. "

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. We split the full document into large "parent" chunks (e.g. 2000 characters).
2. Each parent chunk is further split into smaller "child" chunks (e.g. 400 characters).
3. The child chunks are stored in a VectorStore, while the parent chunks are stored in an in-memory docstore.
4. When we query our Retriever, we do a similarity search comparing our query vector to the child chunks.
5. Instead of returning the child chunks, we return their associated parent chunks.

The basic idea is:

- **Search** for small, focused chunks (better semantic matching)
- **Return** big chunks (richer surrounding context)

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by defining our parent and child splitters.

In [28]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [29]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="wellness_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="wellness_parent_child", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [30]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [31]:
parent_document_retriever.add_documents(raw_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [32]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [33]:
parent_document_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"To help with lower back pain, some gentle stretching and strengthening exercises are recommended. These include:\n\n- **Cat-Cow Stretch:** On hands and knees, alternate arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over your chest, tighten your stomach muscles, and lift your shoulders off the floor. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis slightly. Hold for 10 seconds, then repeat 8-12 times.\n\nThese exercises can help alleviate discomfort and pre

In [34]:
parent_document_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. It is crucial for physical recovery, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep—typically 7 to 9 hours for adults—supports a healthy immune system, maintains optimal brain function, and helps regulate physical and emotional health. Poor sleep or insufficient sleep can lead to negative effects such as decreased energy, impaired concentration, weakened immune function, and increased risk for chronic conditions. Therefore, maintaining good sleep habits and hygiene is essential for overall wellness.'

In [35]:
parent_document_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing deep breathing exercises, engaging in mindfulness or meditation, doing gentle stretching or yoga, taking warm baths, listening to calming music, and using relaxation techniques like progressive muscle relaxation. For headaches specifically, remedies such as staying well-hydrated, applying warm or cold compresses to the head or neck, resting in a dark, quiet room, and massaging your temples or neck can also be helpful. Additionally, essential oils like peppermint or lavender may assist in alleviating headache symptoms.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [36]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [37]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [38]:
ensemble_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, then alternate between arching your back up (cat position) and letting it sag down (cow position). Perform 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for about 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your lower back against the floor by tightening abdominal muscles and tilting your pelvis upward slightly. Hold for 10 seconds and repeat 8-12 times.\n- **Partial Crunches:** Lie on your back with knees bent, cross your arms over your chest, tighten your stomach muscles, and lift your shoulders off the ground briefly, then lower back down. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee towards your chest while keeping the other foot flat on the floor. Hold for 15-30 seconds, then switc

In [39]:
ensemble_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep significantly impacts overall health in multiple ways. Adequate sleep (7-9 hours per night) allows the body to repair tissues, supports immune function, and regulates hormones responsible for growth and appetite. During sleep, the brain also consolidates memories and learns, which is essential for mental clarity and cognitive function. Poor sleep or sleep disorders, such as insomnia, can lead to a range of health issues including increased stress, impaired immune response, mood disturbances, and cognitive decline. Creating a healthy sleep environment and maintaining good sleep hygiene—such as consistent routines, a comfortable environment, and limiting screen time before bed—can promote better sleep quality and, consequently, overall health.'

In [40]:
ensemble_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress include deep breathing exercises, progressive muscle relaxation, grounding techniques such as naming things you see or hear, taking short walks especially in nature, and listening to calming music. For headaches, natural remedies include drinking plenty of water to stay hydrated, applying cold or warm compresses to the head or neck, resting in a dark and quiet room, massaging the temples and neck gently, and using essential oils like peppermint or lavender.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [41]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [42]:
semantic_documents = semantic_chunker.split_documents(raw_docs)

Let's create a new vector store.

In [43]:
semantic_vectorstore = QdrantVectorStore.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide_semantic_chunks"
)

We'll use naive retrieval for this example.

In [44]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [45]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [46]:
semantic_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back upward (cat) and letting it sag down (cow). Perform 10-15 repetitions.\n- Partial Crunches: Lie on your back with knees bent, cross arms over your chest, tighten your stomach muscles, and lift your shoulders off the floor. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds and switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis upward. Hold for 10 seconds and repeat 8-12 times.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged, hold for 5 seconds, then switch sides.\n\nThese gentle stretching and strengthening exercises can alleviate discomfort and help prevent future episodes of lower back pain.'

In [47]:
semantic_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical, mental, and cognitive functions. During sleep, your body undertakes tissue repair, hormone regulation (including growth and appetite hormones), and memory consolidation. Adequate and quality sleep—generally 7-9 hours per night for adults—helps maintain immune function, supports mental well-being, reduces the risk of chronic diseases, and enhances cognitive performance. Poor sleep or sleep disturbances can impair these processes, leading to increased fatigue, weakened immunity, mental health issues, and a higher risk for health problems over time.'

In [48]:
semantic_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- For stress:\n  - Deep breathing exercises (e.g., inhale for 4 counts, hold for 4, exhale for 4)\n  - Progressive muscle relaxation (tensing and releasing muscle groups)\n  - Grounding techniques (naming things you see, hear, feel, smell, and taste)\n  - Short walks in nature\n  - Listening to calming music\n  - Practicing mindfulness and meditation regularly\n\n- For headaches:\n  - Drinking plenty of water to stay hydrated\n  - Applying cold or warm compresses to the head or neck\n  - Resting in a dark, quiet room\n  - Gentle massage of temples and neck\n  - Using peppermint or lavender essential oils\n  - Maintaining a regular sleep schedule\n  - Avoiding known triggers such as dehydration, stress, skipping meals, or certain foods\n\nEngaging in relaxation techniques, ensuring proper hydration, and taking measures to reduce tension can help alleviate headaches and manage stress naturally.'

### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### Answer:

If sentences are short and highly repetitive (like FAQs), semantic chunking can behave poorly because many adjacent sentences have very similar embeddings.

The chunker may merge multiple Q/A items into excessively large, generic chunks.

To address this, we can use a stricter breakpoint threshold so that splitting occurs only when there is a clear topic shift.

It is also better to first split the text based on structure (question/answer pairs) before applying semantic grouping, and then optionally merge chunks if necessary.

---

# 🤝 Breakout Room Part #2

### 🏗️ Activity #1:

Your task is to evaluate the various Retriever methods against each other.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [89]:
import os
import getpass
from uuid import uuid4

os.environ["LANGCHAIN_TRACING_V2"] = "true"

# LangSmith API key (use same value for both to be safe)
ls_key = getpass.getpass("LangSmith API Key: ")
os.environ["LANGCHAIN_API_KEY"] = ls_key

# Project name (unique)
os.environ["LANGCHAIN_PROJECT"] = f"lab11-retriever-eval-{uuid4().hex[:8]}"

# OpenAI key
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")



In [90]:
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# 1) Wrap LLM + embeddings for ragas
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(model="text-embedding-3-small")
)

# 2) Create generator
generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)

# 3) Generate dataset
testset = generator.generate_with_langchain_docs(raw_docs, testset_size=30)

df_test = testset.to_pandas()
df_test.head()

C:\Users\mjelic\AppData\Local\Temp\ipykernel_5832\1982886585.py:8: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
C:\Users\mjelic\AppData\Local\Temp\ipykernel_5832\1982886585.py:9: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(


Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/5 [00:00<?, ?it/s]

...

Skipping multi_hop_abstract_query_synthesizer due to unexpected error: No relationships match the provided condition. Cannot form clusters.


Generating personas:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/31 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,What is covered in Chapter 1 of the exercise g...,[PART 1: EXERCISE AND MOVEMENT\n\nChapter 1: U...,Chapter 1: Understanding Exercise Basics expla...,Holistic Wellness Enthusiast,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer
1,As a Holistic Wellness Enthusiast seeking to i...,[PART 1: EXERCISE AND MOVEMENT\n\nChapter 1: U...,Exercise is one of the most important things y...,Holistic Wellness Enthusiast,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
2,What is PART 1 in exercise?,[PART 1: EXERCISE AND MOVEMENT\n\nChapter 1: U...,"PART 1 refers to exercise and movement, includ...",Holistic Wellness Enthusiast,MISSPELLED,SHORT,single_hop_specific_query_synthesizer
3,"So like, Asia is like a big place right and wh...",[PART 1: EXERCISE AND MOVEMENT\n\nChapter 1: U...,The context provided does not include specific...,Holistic Wellness Enthusiast,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
4,What minerals are important for health and whe...,[PART 2: NUTRITION AND DIET\n\nChapter 4: Fund...,"Minerals are inorganic elements like calcium, ...",Holistic Wellness Enthusiast,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer


In [91]:
from ragas import evaluate
from ragas.metrics import context_precision, context_recall
from datasets import Dataset

df_eval = df_test.sample(10, random_state=42)  

def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

Q_COL  = pick_col(df_test, ["question", "user_input", "query", "prompt"])
GT_COL = pick_col(df_test, ["ground_truth", "reference", "answer", "response", "ground_truths"])

print("Detected columns ->",
      "Q_COL:", Q_COL,
      "| GT_COL:", GT_COL)

def build_ragas_dataset(df, retriever, k=10):
    if Q_COL is None:
        raise ValueError(f"Could not find a question column. Available columns: {list(df.columns)}")
    if GT_COL is None:
        # We can still evaluate precision/recall using contexts only, but Ragas recall needs GT.
        raise ValueError(f"Could not find a ground-truth/answer column. Available columns: {list(df.columns)}")

    rows = []
    for _, row in df.iterrows():
        question = row[Q_COL]

        gt_val = row[GT_COL]
        # sometimes GT is a list; convert to string or pick first
        if isinstance(gt_val, list):
            ground_truth = gt_val[0] if gt_val else ""
        else:
            ground_truth = str(gt_val)

        docs = retriever.get_relevant_documents(question)[:k]
        rows.append({
            "question": question,
            "ground_truth": ground_truth,
            "contexts": [d.page_content for d in docs],
        })

    return Dataset.from_list(rows)


Detected columns -> Q_COL: user_input | GT_COL: reference


C:\Users\mjelic\AppData\Local\Temp\ipykernel_5832\2787371568.py:2: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision, context_recall
C:\Users\mjelic\AppData\Local\Temp\ipykernel_5832\2787371568.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_precision, context_recall


In [92]:
import time
from ragas import evaluate
from ragas.metrics import context_precision, context_recall

metrics = [context_precision, context_recall]


retrievers = {
    "bm25": bm25_retriever,
    "vector_naive": naive_retriever,
    "parent_doc": parent_document_retriever,
    "multi_query": multi_query_retriever,         # OpenAI calls (not Cohere), still can be costly/slow
    "compression": compression_retriever,         # Cohere calls (rate-limited on trial key)
    "ensemble": ensemble_retriever,               # includes compression => Cohere rate limit applies
}

# Which ones need a cooldown because they trigger Cohere rerank:
cohere_limited = {"compression", "ensemble"}

def is_cohere_429(err: Exception) -> bool:
    # Works with different exception types; checks message content
    msg = str(err).lower()
    return ("toomanyrequests" in msg) or ("status_code: 429" in msg) or ("limited to 10 api calls / minute" in msg)

def evaluate_one(name, retriever, k=10, max_retries=3):
    attempt = 0
    while True:
        try:
            ds = build_ragas_dataset(df_eval, retriever, k=k)
            scored = evaluate(ds, metrics=metrics)
            return scored
        except Exception as e:
            attempt += 1
            if is_cohere_429(e) and attempt <= max_retries:
                # wait long enough to clear Cohere trial window
                wait_s = 65
                print(f"[{name}] Hit Cohere 429 rate limit. Sleeping {wait_s}s then retry ({attempt}/{max_retries})...")
                time.sleep(wait_s)
                continue
            raise  # re-raise other errors

results = {}

for name, retriever in retrievers.items():
    print(f"\n=== Evaluating: {name} ===")
    scored = evaluate_one(name, retriever, k=10, max_retries=3)

    # Average metrics
    mean_scores = scored.to_pandas().mean(numeric_only=True).to_dict()
    results[name] = mean_scores
    print("Mean:", mean_scores)

    # Cooldown AFTER running retrievers that use Cohere rerank (and ensemble that includes it)
    if name in cohere_limited:
        print(f"[{name}] Cooldown sleep 65s to avoid Cohere trial limit...")
        time.sleep(65)

print("\nFINAL RESULTS:")
print(results)


C:\Users\mjelic\AppData\Local\Temp\ipykernel_5832\95934792.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision, context_recall
C:\Users\mjelic\AppData\Local\Temp\ipykernel_5832\95934792.py:3: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_precision, context_recall



=== Evaluating: bm25 ===


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Mean: {'context_precision': 0.42499999996125004, 'context_recall': 0.16999999999999998}

=== Evaluating: vector_naive ===


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

...

Mean: {'context_precision': 0.7716468253559086, 'context_recall': 0.7566666666666666}

=== Evaluating: compression ===


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Mean: {'context_precision': 0.9416666666145833, 'context_recall': 0.6983333333333334}
[compression] Cooldown sleep 65s to avoid Cohere trial limit...

=== Evaluating: ensemble ===


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Mean: {'context_precision': 0.9158333332860028, 'context_recall': 0.6566666666666666}
[ensemble] Cooldown sleep 65s to avoid Cohere trial limit...

FINAL RESULTS:
{'bm25': {'context_precision': 0.42499999996125004, 'context_recall': 0.16999999999999998}, 'vector_naive': {'context_precision': 0.6882738094916607, 'context_recall': 0.6233333333333333}, 'parent_doc': {'context_precision': 0.9999999999283334, 'context_recall': 0.8400000000000001}, 'multi_query': {'context_precision': 0.7716468253559086, 'context_recall': 0.7566666666666666}, 'compression': {'context_precision': 0.9416666666145833, 'context_recall': 0.6983333333333334}, 'ensemble': {'context_precision': 0.9158333332860028, 'context_recall': 0.6566666666666666}}


BM25
Performance: Low precision (0.43) and very low recall (0.17). 
Cost: Minimal (no embeddings, no LLM calls).
Latency: Very fast.


Naive Vector Retriever
Performance: Balanced precision (0.69) and recall (0.62).
Cost: Moderate (query embedding per request).
Latency: Moderate.



ParentDocumentRetriever
Performance: Highest precision (~1.00) and highest recall (0.84).
Cost: Slightly higher than naive vector (retrieves parent chunks).
Latency: Slightly higher but still efficient.



MultiQueryRetriever
Performance: High recall (0.76) and strong precision (0.77).
Cost: High (multiple LLM reformulations per query).
Latency: High due to multiple retrieval passes.



Compression Retriever (Cohere Rerank)
Performance: Very high precision (0.94) but moderate recall (0.70).
Cost: High (external reranking API calls).
Latency: High due to reranking step.



Ensemble Retriever
Performance: Strong precision (0.92) but lower recall (0.66) than ParentDoc and MultiQuery.
Cost: High (combines multiple retrievers).
Latency: High.


For this  dataset, ParentDocumentRetriever performs best overall, achieving the highest precision and recall while maintaining reasonable cost and latency.